In [58]:
import pandas as pd
import sqlalchemy
import pyarrow as pa

In [59]:
import pyarrow as pa
print(pa.__version__)

24.0.0


In [60]:
from pathlib import Path

candidate_paths = [
    Path("/usr/local/airflow/dags/raw_data"),
    Path.cwd() / "dags" / "raw_data",
    Path("dags/raw_data"),
    Path("../raw_data"),
]

RAW_DATA = next((p for p in candidate_paths if p.exists()), None)
if RAW_DATA is None:
    raise FileNotFoundError("Could not find raw_data directory")

date_dtype = pd.ArrowDtype(pa.date64())

stock_dfs = {}
for csv_path in sorted(RAW_DATA.glob("*.csv")):
    ticker = csv_path.stem
    df = pd.read_csv(csv_path)
    df["Date"] = pd.to_datetime(df["Date"]).astype(date_dtype)

    dup = df.duplicated(keep=False).sum()
    numeric = df.drop(columns=["Date"])
    neg_count = (numeric <= 0).sum()
    null_count = df.isnull().sum()

    stock_dfs[ticker] = df
    print(f"=== {ticker} ===")
    print(f"  rows={len(df)}, duplicates={dup}")
    print(f"  nulls:", null_count.to_dict())
    print(f"  <=0 counts:", neg_count.to_dict())
    print()

if not stock_dfs:
    raise ValueError(f"No CSV files found in {RAW_DATA}")


=== AAPL ===
  rows=1592, duplicates=0
  nulls: {'Date': 0, 'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}
  <=0 counts: {'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}

=== AMD ===
  rows=1592, duplicates=0
  nulls: {'Date': 0, 'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}
  <=0 counts: {'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}

=== AMZN ===
  rows=1592, duplicates=0
  nulls: {'Date': 0, 'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}
  <=0 counts: {'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}

=== GOOGL ===
  rows=1592, duplicates=0
  nulls: {'Date': 0, 'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}
  <=0 counts: {'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}

=== META ===
  rows=1592, duplicates=0
  nulls: {'Date': 0, 'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}
  <=0 counts: {'Close': 0, 'High': 0, 'Low': 0, 'Open': 0, 'Volume': 0}

=== MSFT ===
  rows=1592, duplicates=0
  nulls: {'

In [61]:

sorted(stock_dfs.keys())

['AAPL',
 'AMD',
 'AMZN',
 'GOOGL',
 'META',
 'MSFT',
 'NFLX',
 'NVDA',
 'TEAM',
 'TSLA']

## Transformation

In [62]:

for ticker, df in stock_dfs.items():
    df['pct_change']    = df['Close'].pct_change() * 100
    df['price_range']   = df['High'] - df['Low']
    df['Moving_Average']          = df['Close'].rolling(window=7).mean() 
    df['daily_return']  = df['Close'].pct_change()
    df['volatility']  = df['daily_return'].rolling(window=7).std()
    
    df.fillna(0,inplace=True)
    stock_dfs[ticker] = df  # save changes back

In [63]:
dfs = []
for ticker, df in stock_dfs.items():
    df['ticker'] = ticker    # add ticker column from the dictionary key
    dfs.append(df)

# Combine all into ONE table
fact_stock_prices = pd.concat(dfs, ignore_index=True)
print(fact_stock_prices.head())
print(fact_stock_prices['ticker'].unique())  # verify all tickers are there

         Date      Close       High        Low       Open     Volume  \
0  2020-01-02  72.400513  72.460776  71.156674  71.409778  135480400   
1  2020-01-03  71.696640  72.455958  71.472462  71.629145  146322800   
2  2020-01-06  72.267929  72.306499  70.568503  70.819201  118387200   
3  2020-01-07  71.928055  72.533095  71.708695  72.277578  108872000   
4  2020-01-08  73.085091  73.386408  71.631537  71.631537  132079200   

   pct_change  price_range  Moving_Average  daily_return  volatility ticker  
0    0.000000     1.304102             0.0      0.000000         0.0   AAPL  
1   -0.972193     0.983496             0.0     -0.009722         0.0   AAPL  
2    0.796814     1.737996             0.0      0.007968         0.0   AAPL  
3   -0.470298     0.824400             0.0     -0.004703         0.0   AAPL  
4    1.608602     1.754871             0.0      0.016086         0.0   AAPL  
<ArrowStringArray>
['AAPL', 'AMD', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NFLX', 'NVDA', 'TEAM',
 'TSLA'

In [64]:
# Fix the date column
fact_stock_prices['Date'] = pd.to_datetime(fact_stock_prices['Date'])

# Verify it changed
print(fact_stock_prices.dtypes)

Date              datetime64[s]
Close                   float64
High                    float64
Low                     float64
Open                    float64
Volume                    int64
pct_change              float64
price_range             float64
Moving_Average          float64
daily_return            float64
volatility              float64
ticker                      str
dtype: object


In [65]:
fact_stock_prices = fact_stock_prices.rename(columns={
    'Date'   : 'date',
    'Open'   : 'open',
    'High'   : 'high',
    'Low'    : 'low',
    'Close'  : 'close',
    'Volume' : 'volume',
    'Moving_Average':'moving_average',
    
})

In [66]:
fact_stock_prices.head()

,date,close,high,low,open,volume,pct_change,price_range,moving_average,daily_return,volatility,ticker
0,2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400,0.000000,1.304102,0.0,0.000000,0.0,AAPL
1,2020-01-03,71.696640,72.455958,71.472462,71.629145,146322800,-0.972193,0.983496,0.0,-0.009722,0.0,AAPL
2,2020-01-06,72.267929,72.306499,70.568503,70.819201,118387200,0.796814,1.737996,0.0,0.007968,0.0,AAPL
3,2020-01-07,71.928055,72.533095,71.708695,72.277578,108872000,-0.470298,0.824400,0.0,-0.004703,0.0,AAPL
4,2020-01-08,73.085091,73.386408,71.631537,71.631537,132079200,1.608602,1.754871,0.0,0.016086,0.0,AAPL


In [67]:
fact_stock_prices.info()

<class 'pandas.DataFrame'>
RangeIndex: 15920 entries, 0 to 15919
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype        
---  ------          --------------  -----        
 0   date            15920 non-null  datetime64[s]
 1   close           15920 non-null  float64      
 2   high            15920 non-null  float64      
 3   low             15920 non-null  float64      
 4   open            15920 non-null  float64      
 5   volume          15920 non-null  int64        
 6   pct_change      15920 non-null  float64      
 7   price_range     15920 non-null  float64      
 8   moving_average  15920 non-null  float64      
 9   daily_return    15920 non-null  float64      
 10  volatility      15920 non-null  float64      
 11  ticker          15920 non-null  str          
dtypes: datetime64[s](1), float64(9), int64(1), str(1)
memory usage: 1.5 MB


## Loading


In [ ]:
import os
from sqlalchemy import create_engine, text

db_user = os.getenv("POSTGRES_USER", "postgres")
db_password = os.getenv("POSTGRES_PASSWORD", "postgres123")
db_names = [name for name in [os.getenv("POSTGRES_DB"), "stock", "mydb"] if name]

connection_candidates = []
database_url = os.getenv("DATABASE_URL")
if database_url:
    connection_candidates.append((database_url, "DATABASE_URL"))

env_host = os.getenv("POSTGRES_HOST")
env_port = os.getenv("POSTGRES_PORT")
if env_host and env_port:
    for db_name in db_names:
        connection_candidates.append((
            f"postgresql://{db_user}:{db_password}@{env_host}:{env_port}/{db_name}",
            f"{env_host}:{env_port}/{db_name}",
        ))

for host, port in [("postgres", "5432"), ("host.docker.internal", "5433"), ("localhost", "5433")]:
    for db_name in db_names:
        connection_candidates.append((
            f"postgresql://{db_user}:{db_password}@{host}:{port}/{db_name}",
            f"{host}:{port}/{db_name}",
        ))

engine = None
last_error = None
seen = set()

for url, label in connection_candidates:
    if url in seen:
        continue
    seen.add(url)
    try:
        candidate_engine = create_engine(url)
        with candidate_engine.begin() as conn:
            conn.execute(text("SELECT 1"))
            conn.execute(text("CREATE SCHEMA IF NOT EXISTS dwh"))
        engine = candidate_engine
        print(f"Connected to PostgreSQL at {label}")
        break
    except Exception as exc:
        last_error = exc

if engine is None:
    raise ConnectionError(
        "Could not connect to PostgreSQL. Set DATABASE_URL or POSTGRES_HOST/POSTGRES_PORT/POSTGRES_DB. "
        "If Postgres is running via docker with -p 5433:5432, use host.docker.internal:5433 from inside Airflow containers."
    ) from last_error

fact_stock_prices.to_sql('fact_stock_prices', engine, schema='dwh', if_exists='replace', index=False)
print("✅ fact_stock_prices uploaded!")

✅ fact_stock_prices uploaded!
